In [1]:
import pandas as pd
import numpy as np
import holidays
import os

In [2]:
# 1. Verification Gate: Ensure Phase 1 grounding truth is present
registry_file = "location_registry.csv"
if not os.path.exists(registry_file):
    raise FileNotFoundError(f"Missing master registry! Please run Notebook 01A first.")

df_registry = pd.read_csv(registry_file)

In [3]:
# 2. Programmatically generate the 5-year timeline sequence (1,826 days total)
date_range = pd.date_range(start='2021-01-01', end='2025-12-31', freq='D')
df_dates = pd.DataFrame({'date': date_range})
df_dates['date'] = df_dates['date'].dt.strftime('%Y-%m-%d')

In [4]:
# 3. Create a cross-join matrix (every date matched with every location)
df_dates['key'] = 1
df_registry['key'] = 1
df_crowd = pd.merge(df_dates, df_registry, on='key').drop(columns=['key'])

In [5]:
print("--- NOTEBOOK 03 BASE MATRIX CREATED ---")
print(f"Verified DataFrame Shape: {df_crowd.shape} (Expected: 18260 rows)")

--- NOTEBOOK 03 BASE MATRIX CREATED ---
Verified DataFrame Shape: (18260, 12) (Expected: 18260 rows)


In [6]:
# Retain structural data assets
base_columns = ['date', 'location_id', 'location', 'destination_type', 'tourism_popularity_rating']
df_crowd = df_crowd[base_columns]
df_crowd.head(3)

,date,location_id,location,destination_type,tourism_popularity_rating
0,2021-01-01,LOC001,Manali,Mountain Hill Station,4.8
1,2021-01-01,LOC002,Shimla,Mountain Hill Station,4.8
2,2021-01-01,LOC003,Mussoorie,Mountain Hill Station,4.8


## Calendar Intelligence & Advanced Indian Festival Weights

In [7]:
# 1. Date Splitting
df_crowd['date_parsed'] = pd.to_datetime(df_crowd['date'])
df_crowd['month'] = df_crowd['date_parsed'].dt.month
df_crowd['day_of_week'] = df_crowd['date_parsed'].dt.dayofweek
df_crowd['is_weekend'] = df_crowd['day_of_week'].isin([5, 6]).astype(int)

In [8]:
# 2. Ingest Indian National & Gazetted Holiday Calendar
india_holiday_registry = holidays.India(years=list(range(2021, 2026)))
df_crowd['is_holiday'] = df_crowd['date'].apply(lambda d: 1 if d in india_holiday_registry else 0)
df_crowd['holiday_name'] = df_crowd['date'].apply(lambda d: india_holiday_registry.get(d) if d in india_holiday_registry else "None")

In [9]:
# 3. UPGRADE: Granular Festival Boost Engine (Prevents unweighted single-day biases)
festival_boost_map = {
    "Republic Day": 5, "Independence Day": 5, "Gandhi Jayanti": 5,
    "Good Friday": 5, "Holi": 8, "Maha Shivaratri": 5
}
df_crowd['festival_boost'] = df_crowd['holiday_name'].map(festival_boost_map).fillna(0.0)

In [10]:
# Catch floating or multi-day massive festive surges via string containment matches
df_crowd.loc[df_crowd['holiday_name'].str.contains('Diwali|Deepavali', case=False, na=False), 'festival_boost'] = 15.0
df_crowd.loc[df_crowd['holiday_name'].str.contains('Christmas', case=False, na=False), 'festival_boost'] = 10.0
df_crowd.loc[df_crowd['holiday_name'].str.contains('New Year', case=False, na=False), 'festival_boost'] = 15.0

In [11]:
# 4. Long Weekend Clustering Flag
df_crowd['is_day_off'] = ((df_crowd['is_weekend'] == 1) | (df_crowd['is_holiday'] == 1)).astype(int)
df_crowd['block'] = (df_crowd['is_day_off'] != df_crowd.groupby('location')['is_day_off'].shift()).cumsum()
block_sizes = df_crowd.groupby(['location', 'block'])['is_day_off'].transform('sum')
df_crowd['long_weekend_flag'] = ((df_crowd['is_day_off'] == 1) & (block_sizes >= 3)).astype(int)

In [12]:
# 5. Cyclical Month Encodings for ML Generalization
df_crowd['month_sin'] = np.sin(2 * np.pi * df_crowd['month'] / 12.0)
df_crowd['month_cos'] = np.cos(2 * np.pi * df_crowd['month'] / 12.0)

df_crowd = df_crowd.drop(columns=['date_parsed', 'is_day_off', 'block'])
print("Calendar intelligence and weighted festival indices generated successfully.")

Calendar intelligence and weighted festival indices generated successfully.


## Environment-Driven Seasonality & Clean Popularity Scaling

In [13]:
# 1.  Indian School Summer Vacation Flag (May and June migration)
df_crowd['school_vacation_flag'] = df_crowd['month'].isin([5, 6]).astype(int)

In [14]:
# 2. Geographic Peak Month Mapping
peak_months_map = {
    "Mountain Hill Station": [4, 5, 6, 10],      # Apr, May, Jun, Oct
    "Coastal Plain": [11, 12, 1, 2],             # Nov, Dec, Jan, Feb
    "High Altitude Desert": [6, 7, 8, 9],        # Jun, Jul, Aug, Sep
    "Semi-Arid Plain": [11, 12, 1]               # Nov, Dec, Jan
}

In [15]:
df_crowd['seasonal_tourism_score'] = df_crowd.apply(
    lambda r: 1.00 if r['month'] in peak_months_map.get(r['destination_type'], []) else 0.25, 
    axis=1
)

In [16]:
# 3.  Pure Linear Popularity Scaling (Wipes out accordion distortion)
df_crowd['normalized_popularity'] = df_crowd['tourism_popularity_rating'] / 5.0

print("School vacation indicators, regional seasonality, and linear scaling built cleanly.")

School vacation indicators, regional seasonality, and linear scaling built cleanly.


## Production Point Contribution Engine, Sanity Checks & Export

In [17]:
# 1. Initialize Point Accumulation Framework
df_crowd['crowd_score'] = 0.0

In [18]:
# Human Mobility Indicators
df_crowd.loc[df_crowd['is_weekend'] == 1, 'crowd_score'] += 20
df_crowd.loc[df_crowd['is_holiday'] == 1, 'crowd_score'] += 20
df_crowd.loc[df_crowd['long_weekend_flag'] == 1, 'crowd_score'] += 15
df_crowd.loc[df_crowd['school_vacation_flag'] == 1, 'crowd_score'] += 20

In [19]:
# Add Variable Festival Surge Contribution
df_crowd['crowd_score'] += df_crowd['festival_boost']

# Background Environment Indicators
df_crowd['crowd_score'] += df_crowd['seasonal_tourism_score'] * 15
df_crowd['crowd_score'] += df_crowd['normalized_popularity'] * 10

In [20]:
# 2. Bind strict maximum boundary threshold ceiling
df_crowd['crowd_baseline'] = df_crowd['crowd_score'].clip(upper=100.0)

# Drop raw indexing markers before storage
df_crowd_export = df_crowd.drop(columns=['crowd_score', 'month'])

# Export finalized asset directly to disk
df_crowd_export.to_csv("crowd_features.csv", index=False)
print(" Notebook 03 Complete! Standalone matrix written to 'crowd_features.csv'\n")

 Notebook 03 Complete! Standalone matrix written to 'crowd_features.csv'



In [21]:
# =====================================================================
# MANDATORY PROJECTS ARCHITECTURE SANITY CHECKS
# =====================================================================
print("=====================================================================")
print("PORTFOLIO SANITY CHECK 1: MEAN CROWD BASELINE BY LOCATION")
print("=====================================================================")
print(df_crowd.groupby('location')['crowd_baseline'].mean().sort_values(ascending=False))

print("\n=====================================================================")
print("PORTFOLIO SANITY CHECK 2: MEAN CROWD BASELINE BY CHRONOLOGICAL MONTH")
print("=====================================================================")
print(df_crowd.groupby('month')['crowd_baseline'].mean())

PORTFOLIO SANITY CHECK 1: MEAN CROWD BASELINE BY LOCATION
location
Goa           27.334912
Ooty          27.288609
Darjeeling    27.190361
Manali        27.190361
Mussoorie     27.190361
Shimla        27.190361
Leh           26.990361
Nainital      26.390361
Jaipur        26.266210
Munnar        25.990361
Name: crowd_baseline, dtype: float64

PORTFOLIO SANITY CHECK 2: MEAN CROWD BASELINE BY CHRONOLOGICAL MONTH
month
1     21.981935
2     20.243369
3     19.628710
4     28.698333
5     47.661774
6     48.253333
7     20.666613
8     21.647258
9     20.418333
10    29.103710
11    22.296667
12    22.014194
Name: crowd_baseline, dtype: float64
